In [ ]:
"""
03_extract_market_data.py
=========================
Download daily S&P 500 (^GSPC) data via yfinance.

No API key required. Data includes:
  - Adjusted closing prices
  - Daily returns
  - Realised 5-day & 21-day rolling volatility (as VIX complement)
  - Forward 1-day and 5-day S&P 500 returns (dependent variables for models)

Dependencies: pip install yfinance pandas
"""

import os
import numpy as np
import pandas as pd
import yfinance as yf

START_DATE = "2022-01-01"
END_DATE   = "2026-04-14"

# ── Market data ────────────────────────────────────────────────────────────
# Downloaded via yfinance – no API key needed.
SP500_TICKER = "^GSPC"

# ── Output paths ───────────────────────────────────────────────────────────
DATA_DIR    = "C:/Python/CSUREMM"
OUTPUTS_DIR = "C:/Python/CSUREMM/outputs"

os.makedirs(DATA_DIR, exist_ok=True)


def download(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Download OHLCV + Adj Close for a single ticker."""
    print(f"[yfinance] Downloading {ticker} …", end=" ", flush=True)
    # add one day buffer so we don't lose the last observation to yfinance's
    # exclusive-end convention
    df = yf.download(ticker, start=start, end=end,
                     auto_adjust=True, progress=False)
    df.index = pd.to_datetime(df.index).normalize()
    df.index.name = "date"
    print(f"{len(df)} rows")
    return df


def build_sp500_features(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich the raw S&P 500 DataFrame with return and vol features."""
    out = pd.DataFrame(index=df.index)

    price = df["Close"].squeeze()
    out["sp500_close"]       = price
    out["sp500_return_1d"]   = price.pct_change(1)
    out["sp500_return_5d"]   = price.pct_change(5)
    out["sp500_return_21d"]  = price.pct_change(21)

    # Forward returns (what we ultimately want to predict / correlate)
    out["sp500_fwd_1d"]      = price.pct_change(1).shift(-1)
    out["sp500_fwd_5d"]      = price.pct_change(5).shift(-5)

    # Realised volatility (annualised)
    log_ret = np.log(price / price.shift(1))
    out["sp500_rv5"]         = log_ret.rolling(5).std()  * np.sqrt(252)
    out["sp500_rv21"]        = log_ret.rolling(21).std() * np.sqrt(252)

    # Price levels for plotting
    out["sp500_open"]        = df["Open"].squeeze()
    out["sp500_high"]        = df["High"].squeeze()
    out["sp500_low"]         = df["Low"].squeeze()
    out["sp500_volume"]      = df["Volume"].squeeze()

    return out
    return out

def main():
    # ── Download raw data ──────────────────────────────────────────────────
    sp500_raw = download(SP500_TICKER, START_DATE, END_DATE)

    # ── Build feature frames ───────────────────────────────────────────────
    sp500 = build_sp500_features(sp500_raw)

    # ── Save individual files ──────────────────────────────────────────────
    sp500.to_csv(f"{DATA_DIR}/sp500_daily.csv")
    print(f"[save] {DATA_DIR}/sp500_daily.csv  ({len(sp500)} rows)")

    # ── Quick sanity check ─────────────────────────────────────────────────
    print("\n── S&P 500 sample ──────────────────────────────────────────────")
    print(sp500[["sp500_close", "sp500_return_1d",
                  "sp500_rv21", "sp500_fwd_1d"]].tail(5).to_string())

if __name__ == "__main__":
    main()

In [ ]:
# Plotting gold prices and SP500 close on the same time series plot with dual y-axes

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Load data ---
gold = pd.read_csv("gold_prices_daily_normalized.csv", parse_dates=["Time"])
sp500 = pd.read_csv("sp500_daily.csv", parse_dates=["date"])

gold = gold.rename(columns={"Time": "date"})
gold = gold.sort_values("date")
sp500 = sp500[sp500["date"] >= "2024-01-01"].sort_values("date")

#  --- Plot ---
sns.set_theme(style="whitegrid")
palette = sns.color_palette()  # default seaborn palette
 
fig, ax1 = plt.subplots(figsize=(14, 6))
 
# SP500 close on left y-axis
ax1.plot(sp500["date"], sp500["sp500_close"], color=palette[0], label="SP500 Close", linewidth=2.5)
ax1.set_xlabel("Date")
ax1.set_ylabel("SP500 Close", color=palette[0])
ax1.tick_params(axis="y", labelcolor=palette[0])
 
# Gold normalized on right y-axis
ax2 = ax1.twinx()
ax2.plot(gold["date"], gold["normalized"], color=palette[1], label="Gold Normalized", linewidth=2.5, alpha=0.85)
ax2.set_ylabel("Gold Normalized Value", color=palette[1])
ax2.tick_params(axis="y", labelcolor=palette[1])
 
# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
 
plt.title("SP500 Close vs Gold Normalized Value")
fig.tight_layout()
plt.savefig("sp500_gold_timeseries.png", dpi=150)
plt.show()